# ANACOR — Análise de Correspondência

Descreve a estrutura da associação entre perfil de desafio e velocidade da equipe em um mapa perceptual (Hair et al., 2009). O qui-quadrado responde *se* existe associação; a ANACOR, *com o que ela se parece*.

**Um mapa só é gerado se:** associação significativa (α = 0,05) · premissa das frequências esperadas atendida · tabela ≥ 3×3, já que admite `mín(linhas−1, colunas−1)` dimensões.

O par aprovado no notebook 02 (**Q17** × curva da equipe) é 2×2 e por isso não gera mapa. A ANACOR usa par próprio, **derivado do achado da bateria**: **perfil de desafio** (**Q17** CI/CD × **Q13** complexidade, 3 níveis) × **Q25** (tempo até produção, 3 faixas). O agrupamento em 3 níveis foi definido para atender à premissa e ao mínimo 3×3, não para maximizar significância — a associação se mantém sob recortes alternativos (p entre 0,0038 e 0,0229).

---
## Configuração

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, utils

utils.configurar_visualizacoes()

# Reaproveita o df do master, ou carrega e trata se rodando isolado
df = utils.garantir_preparado(globals().get('df'))
print(f"{len(df)} respostas × {len(df.columns)} variáveis")

---
## Pares aprovados no qui-quadrado

Verifica quais pares testados no notebook 02 passaram em todos os critérios. Pares 2×2, ainda que significativos, não geram mapa bidimensional.

In [ ]:
# Consome os resultados publicados pelo notebook 02, se disponíveis
resultados_qui = globals().get('resultados_qui', {})

if resultados_qui:
    print(f"Pares testados no notebook 02: {len(resultados_qui)}\n")
    for (v1, v2), r in resultados_qui.items():
        if not r.get('significativo'):
            continue
        t = r['tabela']
        dim = min(t.shape[0] - 1, t.shape[1] - 1)
        print(f"  {r['rotulo']}")
        print(f"    tabela {t.shape[0]}x{t.shape[1]} | dimensões possíveis: {dim}"
              f" | {'gera mapa' if dim >= 2 else 'NÃO gera mapa 2D'}")
else:
    print("Notebook 02 não foi executado nesta sessão; seguindo com o par derivado da bateria.")

---
## Mapa perceptual: perfil de desafio × tempo da equipe

O pré-teste qui-quadrado é executado automaticamente. O mapa só é gerado se todos os critérios forem atendidos.

In [ ]:
resultado_anacor = utils.anacor_condicionada(
    df,
    'perfil_desafio_3',
    'q25_faixa_deploy',
    "Perfil de desafio (Q17 CI/CD × Q13 complexidade) × Tempo da equipe até produção (Q25)",
    salvar_grafico=True
)

---
## Pares que a metodologia previa mas os dados não sustentam

O projeto de pesquisa previa mapear *desafio por porte de empresa* e *estratégia por velocidade*. Os pares são testados abaixo e o resultado — inclusive negativo — é reportado, em vez de omitido.

In [ ]:
# Pares previstos no Projeto de Pesquisa. São testados e o resultado é
# reportado mesmo quando negativo — omiti-los seria seleção de achados.
PARES_PREVISTOS = [
    ('q6_porte_empresa', 'q18_desafio_macro',
     "Porte da empresa (Q6) × Principal desafio (Q18)"),
    ('q22_fonte_c', 'faixa_tempo_individual',
     "Fonte de aprendizado mais útil (Q22) × Tempo individual (Q24)"),
    ('q26_fator_macro', 'q25_faixa_deploy',
     "Fator acelerador (Q26) × Tempo da equipe (Q25)"),
]

resultados_previstos = {}
for v1, v2, rotulo in PARES_PREVISTOS:
    resultados_previstos[rotulo] = utils.anacor_condicionada(
        df, v1, v2, rotulo, salvar_grafico=True)

---
## Sensibilidade: fonte de aprendizado em 3 níveis

O par *Fonte mais útil × Tempo individual* é recusado por premissa (5×3, 40% das células com esperado < 5). Para distinguir esparsidade de ausência de associação, a fonte é reagrupada em 3 níveis (autoestudo · treinamento formal · social/prática) e o par é retestado. Reportado como análise de sensibilidade do par previsto, não como novo par.

In [ ]:
resultado_fonte3 = utils.anacor_condicionada(
    df, 'q22_fonte_3', 'faixa_tempo_individual',
    "Sensibilidade: Fonte mais útil em 3 níveis (Q22) × Tempo individual (Q24)",
    salvar_grafico=True)

## Como ler o mapa

Categorias próximas estão associadas; a distância ao centro indica afastamento do perfil médio. A leitura mais informativa é por dimensão: se as faixas de tempo aparecem ordenadas ao longo de um eixo, esse eixo é o gradiente de velocidade.

**Ressalvas:** proximidade entre linha e coluna é qualitativa, não métrica · acumulado de 100% não indica bom ajuste, apenas que a tabela admite exatamente 2 dimensões · a técnica é descritiva e o desenho transversal não permite afirmar causalidade.

In [ ]:
utils.titulo_secao("ANACOR Concluída")

todos = {"Perfil de desafio × Tempo da equipe": resultado_anacor, **resultados_previstos,
         "Sensibilidade: Fonte em 3 níveis × Tempo individual": resultado_fonte3}
aplicadas = [k for k, v in todos.items() if v['aplicada']]

print(f"\nPares avaliados: {len(todos)} | mapas gerados: {len(aplicadas)}\n")
for k, v in todos.items():
    if v['aplicada']:
        pct = v['anacor']['pct_variancia']
        print(f"  MAPA     {k}\n           dim1 {pct[0]:.2f}% | dim2 {pct[1]:.2f}%")
    else:
        print(f"  recusado {k}\n           {v['motivo']}")